# Task 2 figures and label audit (CPU only, ~5 minutes)

Attach the **output of the finished Task 2 notebook** as an input (Add Input → Your Work). Its
`results/task2` folder is restored automatically. No GPU is needed: set Accelerator to **None**
and Internet to **on**.

1. **Label audit sheets:** every annotation-derived test pair, with its generated answers and no
   model predictions. Open the PNGs, and in `label_audit.csv` put `y` in the `doubtful(y/n)` column
   for every image whose generated answer you think is wrong or ambiguous. Pairs are numbered as
   on the sheets.
2. **Aligned examples figure** for the report.

In [ ]:
# ---- EDIT THIS: your public GitHub repository ----
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"
BRANCH = "main"

import os, sys, glob, shutil, subprocess, time, json, pathlib
NOTEBOOK_START = time.time()
TEST = os.environ.get("PA3_TEST") == "1"          # only used when testing this notebook on a CPU
if TEST:
    REPO = os.environ["PA3_REPO"]
    OUT = os.environ.get("PA3_OUT", "/tmp/pa3_nbtest")
else:
    REPO = "/tmp/atml-pa3"                          # code: scratch disk, not saved
    OUT = "/kaggle/working"                         # results: saved with the notebook version
    os.environ["HF_HOME"] = "/tmp/hf"               # model/dataset caches: scratch, not saved
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements.txt"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
ROOT = f"{OUT}/results/task2"
os.makedirs(ROOT, exist_ok=True)

# Resume support: attach an earlier version of this notebook (its Output) as an input dataset and
# everything it finished is copied back here, so finished jobs are skipped.
# (Kaggle mounts another notebook's output a few folders deep, e.g.
#  /kaggle/input/notebooks/<user>/<notebook>/results/task2, so search at any depth.)
found = sorted(set(glob.glob(f"/kaggle/input/**/results/task2", recursive=True)))
for prev in found:
    print("restoring finished work from", prev)
    shutil.copytree(prev, ROOT, dirs_exist_ok=True)
if not found and os.path.isdir("/kaggle/input"):
    print("no earlier results/task2 found among the inputs; inputs present:",
          [p for p in glob.glob("/kaggle/input/**", recursive=True) if p.count("/") <= 6][:30])

def sh(cmd):
    """Run a shell command from the repo, streaming its output; stop the notebook if it fails."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
    for line in p.stdout:
        if "Warning" not in line and "warn(" not in line:
            print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")

import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)] or "none (CPU test mode)")
print("commit:", subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
IMAGES = os.environ.get("PA3_FX2_IMAGES", "/tmp/pa3_images")
sh(f"python -m task2.figures --root {ROOT} --images {IMAGES}")
from IPython.display import Image, display
for f in sorted(glob.glob(f"{ROOT}/data/label_audit_*.png")):
    print(f); display(Image(f, width=1100))

In [ ]:
for run in ["run", "run_pool49"]:
    if os.path.exists(f"{ROOT}/{run}/test_predictions.jsonl"):
        sh(f"python -m task2.analyze --root {ROOT} --run {ROOT}/{run} --images {IMAGES} > /dev/null")
display(Image(f"{ROOT}/run/report/fig_examples.png", width=900))

In [ ]:
import zipfile
zp = f"{OUT}/task2_figures.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in list(pathlib.Path(ROOT, "data").glob("label_audit*")) + list(pathlib.Path(ROOT).glob("run*/report/*")):
        z.write(p, p.relative_to(OUT))
print(zp, round(os.path.getsize(zp) / 2**20, 1), "MB")